In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline

In [2]:
df = pd.read_csv("./cardekho_imputated.csv", index_col=0)
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [3]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [4]:
df.drop("car_name", axis=1, inplace=True)
df.drop("brand", axis=1, inplace=True)

In [5]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [6]:
df.model.unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [7]:
## Getting All Different Types OF Features
num_features = [feature for feature in df.columns if df[feature].dtype != "O"]
print("Num of Numerical Features :", len(num_features))

cat_features = [feature for feature in df.columns if df[feature].dtype == "O"]
print("Num of Categorical Features :", len(cat_features))

discrete_features = [
    feature for feature in num_features if len(df[feature].unique()) <= 25
]
print("Num of Discrete Features :", len(discrete_features))

continuous_features = [
    feature for feature in num_features if feature not in discrete_features
]
print("Num of Continuous Features :", len(continuous_features))

Num of Numerical Features : 7
Num of Categorical Features : 4
Num of Discrete Features : 2
Num of Continuous Features : 5


In [8]:
X = df.drop("selling_price", axis=1)
y = df["selling_price"]

### Feature Encoding and Scaling


In [9]:
len(df["model"].unique())

120

In [10]:
df["model"].value_counts()

model
i20            906
Swift Dzire    890
Swift          781
Alto           778
City           757
              ... 
Ghibli           1
Altroz           1
GTC4Lusso        1
Aura             1
Gurkha           1
Name: count, Length: 120, dtype: int64

In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

X["model"] = le.fit_transform(X["model"])

X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [12]:
print(len(df["seller_type"].unique()))
print(len(df["fuel_type"].unique()))
print(len(df["transmission_type"].unique()))

3
5
2


In [13]:
numeric_columns = X.select_dtypes(exclude="O").columns
one_hot_columns = X.select_dtypes(include="O").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop="first")

preprocessor = ColumnTransformer(
    transformers=[
        ("StandardScaler", numeric_transformer, numeric_columns),
        ("OneHotEncoder", categorical_transformer, one_hot_columns),
    ],
    remainder="passthrough",
)

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

### Model Training and Model Selection


In [16]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [17]:
# create a function to evaluate a model
def evaluate_model(true, predicted):
    mse = mean_squared_error(true, predicted)
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(true, predicted)
    return mae, rmse, r2

In [18]:
## Begin the model training

models = {
    "Random Forest": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    # make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    mtr_mae, mtr_rmse, mtr_r2 = evaluate_model(y_train, y_train_pred)
    mte_mae, mte_rmse, mte_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])

    print("Model performance for Training set")
    print("- Root Mean Squared Error: {:.4f}".format(mtr_rmse))
    print("- Mean Absolute Error: {:.4f}".format(mtr_mae))
    print("- R2 Score: {:.4f}".format(mtr_r2))

    print("----------------------------------")

    print("Model performance for Test set")
    print("- Root Mean Squared Error: {:.4f}".format(mte_rmse))
    print("- Mean Absolute Error: {:.4f}".format(mte_mae))
    print("- R2 Score: {:.4f}".format(mte_r2))

    print("=" * 35)
    print("\n")

Random Forest
Model performance for Training set
- Root Mean Squared Error: 144842.2875
- Mean Absolute Error: 40034.8407
- R2 Score: 0.9741
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 230315.6993
- Mean Absolute Error: 102017.6623
- R2 Score: 0.9295


XGBRegressor
Model performance for Training set
- Root Mean Squared Error: 85441.7918
- Mean Absolute Error: 61009.3145
- R2 Score: 0.9910
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 245992.2368
- Mean Absolute Error: 99017.9637
- R2 Score: 0.9196




### Hyper Parameters Tuning


In [20]:
rf_params = {
    "max_depth": [5, 8, 15, None, 10],
    "max_features": [5, 7, "auto", 8],
    "min_samples_split": [2, 8, 15, 20],
    "n_estimators": [100, 200, 500, 1000],
}
xgb_params = {
    "n_estimators": [100, 200, 500, 1000],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.5],
    "max_depth": [3, 5, 8, 10, None],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.5, 0.7, 1.0],
    "gamma": [0, 0.1, 0.3, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
}

In [21]:
randomized_models = [
    ("XGBRegressor", XGBRegressor(), xgb_params),
    ("Random Forest", RandomForestRegressor(), rf_params),
]

In [22]:
from sklearn.model_selection import RandomizedSearchCV

model_params = {}

for model_name, model, params in randomized_models:
    rscv = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=100,
        cv=3,
        verbose=2,
        n_jobs=-1,
    )

    rscv.fit(X_train, y_train)
    model_params[model_name] = rscv.best_params_

for model_name in model_params:
    print("Best Params for: ", model_name)
    print(model_params[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best Params for:  XGBRegressor
{'subsample': 0.6, 'reg_lambda': 2.0, 'reg_alpha': 0.01, 'n_estimators': 1000, 'min_child_weight': 10, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 0.7}
Best Params for:  Random Forest
{'n_estimators': 100, 'min_samples_split': 2, 'max_features': 8, 'max_depth': 15}


In [23]:
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=100, min_samples_split=2, max_features=5, max_depth=None
    ),
    "XGBRegressor": XGBRegressor(
        n_estimators=1000,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.6,
        colsample_bytree=0.7,
        min_child_weight=10,
        gamma=0.3,
        reg_alpha=0.01,
        reg_lambda=2.0,
        random_state=42,
    ),
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    # make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    mtr_mae, mtr_rmse, mtr_r2 = evaluate_model(y_train, y_train_pred)
    mte_mae, mte_rmse, mte_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])

    print("Model performance for Training set")
    print("- Root Mean Squared Error: {:.4f}".format(mtr_rmse))
    print("- Mean Absolute Error: {:.4f}".format(mtr_mae))
    print("- R2 Score: {:.4f}".format(mtr_r2))

    print("----------------------------------")

    print("Model performance for Test set")
    print("- Root Mean Squared Error: {:.4f}".format(mte_rmse))
    print("- Mean Absolute Error: {:.4f}".format(mte_mae))
    print("- R2 Score: {:.4f}".format(mte_r2))

    print("=" * 35)
    print("\n")

Random Forest
Model performance for Training set
- Root Mean Squared Error: 127288.6001
- Mean Absolute Error: 39551.8515
- R2 Score: 0.9800
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 206413.5198
- Mean Absolute Error: 98320.6000
- R2 Score: 0.9434


XGBRegressor
Model performance for Training set
- Root Mean Squared Error: 118665.8476
- Mean Absolute Error: 76863.5598
- R2 Score: 0.9826
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 325854.3919
- Mean Absolute Error: 107967.7309
- R2 Score: 0.8589


